# Before the Breach — Granite historical-neighbor challenger

**Goal:** test a genuinely new signal: is a CVE semantically similar to *already-known-at-the-time* exploited CVEs? Re-rank within the exploit/no-exploit groups and compare to exploit-first → EPSS.

**Key limitation:** the bundled `vulnerabilities.csv` contains **current**, not historical, descriptions. Therefore default runs are **exploratory only** and cannot establish a leakage-free improvement. For a fair submission, replace descriptions with point-in-time archived text before using the challenger. This notebook never uses future KEV membership as a feature, but current descriptions still leak possible later edits.

**Runs:** 2025 Q1–Q3 rolling validation; 2026 candidate ranking. Keeps `top500_baseline.csv` unchanged as safe fallback; writes candidate `top500_granite_neighbors_EXPLORATORY.csv` and `comparison.csv`. No model is guaranteed to beat 8 hits on the hidden 2026 benchmark.

Install in the selected kernel: `%pip install -U pandas numpy scikit-learn sentence-transformers torch`.


In [2]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.stats import rankdata

DATA=Path('data')
OUT=Path('neighbor_outputs'); OUT.mkdir(exist_ok=True)
SEED=42
FLAGS=['exploitdb','metasploit','nuclei','poc_github']
K_FRAC=500/72670
HORIZON=pd.Timedelta(days=90)
CUTOFF=pd.Timestamp('2025-12-31')
for name in ['vulnerabilities.csv','known_exploited.csv','scoreboard_2026.csv','exploit_signals_2024.csv','exploit_signals_2025.csv']:
    assert (DATA/name).exists(), f'Missing data/{name}'
print('Input files found')


Input files found


## 1. Build publication-month candidates and mature future-KEV labels

This matches one candidate per published CVE and removes CVEs that were already in KEV at the end of their snapshot month. February 2025 is excluded because of the documented signal gap. We use `date_added` only as a historical training label or to identify KEV entries already known before a snapshot.


In [3]:
vuln=pd.read_csv(DATA/'vulnerabilities.csv',usecols=['cve_id','published','description'],low_memory=False)
vuln['published']=pd.to_datetime(vuln['published'],errors='coerce')
kev=pd.read_csv(DATA/'known_exploited.csv',usecols=['cve_id','date_added']).drop_duplicates('cve_id')
kev['date_added']=pd.to_datetime(kev.date_added,errors='coerce')
sigs=pd.concat([pd.read_csv(DATA/f'exploit_signals_{yr}.csv',usecols=['month','cve_id','epss',*FLAGS]) for yr in [2024,2025]],ignore_index=True)
sigs=sigs.rename(columns={'month':'snapshot_month'}).drop_duplicates(['cve_id','snapshot_month'])
cohort=vuln[vuln.published.between('2024-01-01','2025-09-30')][['cve_id','published','description']].copy()
cohort['snapshot_month']=cohort.published.dt.strftime('%Y-%m')
cohort=cohort[cohort.snapshot_month!='2025-02']
cohort=cohort.merge(sigs,on=['cve_id','snapshot_month'],how='left',validate='one_to_one').merge(kev,on='cve_id',how='left',validate='one_to_one')
cohort['snapshot_date']=pd.to_datetime(cohort.snapshot_month+'-01')+pd.offsets.MonthEnd()
cohort=cohort[cohort.date_added.isna() | (cohort.date_added>cohort.snapshot_date)].copy()
cohort=cohort[cohort.snapshot_date+HORIZON<=CUTOFF].copy()
cohort['label']=(cohort.date_added.notna() & (cohort.date_added>cohort.snapshot_date) & (cohort.date_added<=cohort.snapshot_date+HORIZON)).astype('int8')
scoreboard=pd.read_csv(DATA/'scoreboard_2026.csv')
scoreboard['published']=pd.to_datetime(scoreboard.published,errors='coerce')
scoreboard['snapshot_date']=pd.to_datetime(scoreboard.snapshot_month+'-01')+pd.offsets.MonthEnd()
scoreboard=scoreboard.merge(vuln[['cve_id','description']],on='cve_id',how='left',validate='one_to_one')
for d in [cohort,scoreboard]:
    d['epss']=d.epss.fillna(0).clip(0,1)
    d[FLAGS]=d[FLAGS].fillna(0).astype('int8')
    d['any_exploit']=d[FLAGS].max(axis=1).astype('int8')
print('Historic rows',len(cohort),'positive',cohort.label.sum(),'scoreboard',len(scoreboard))


Historic rows 71485 positive 70 scoreboard 72670


## 2. Encode descriptions once, with disk cache

**EXPLORATORY CURRENT-DESCRIPTION MODE**. These texts can have post-snapshot modifications. Before using the method in a contest submission, construct archived descriptions from historical releases of [cvelistV5](https://github.com/CVEProject/cvelistV5) for each historical snapshot. Re-running with latest descriptions is not a substitute for temporal data provenance.

CPU encoding may take substantial time on the first run. Cache is tied to the exact sorted CVE/text inventory hash so stale caches are not silently mixed with changed texts.


In [ ]:
%pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

needed_ids=set(cohort.cve_id)|set(scoreboard.cve_id)|set(kev.cve_id)
all_text=vuln[vuln.cve_id.isin(needed_ids)][['cve_id','description']].drop_duplicates('cve_id').sort_values('cve_id').reset_index(drop=True)
all_text['description']=all_text.description.fillna('').astype(str).str.slice(0,1600)
key=hashlib.sha256(pd.util.hash_pandas_object(all_text[['cve_id','description']],index=False).values.tobytes()).hexdigest()[:16]
cache=OUT/f'granite_30m_{key}.npz'
if cache.exists():
    from_cache=np.load(cache)
    vec=from_cache['embeddings']
    print('Loaded embeddings',cache,vec.shape)
else:
    encoder=SentenceTransformer('ibm-granite/granite-embedding-30m-english')
    vec=encoder.encode(all_text.description.tolist(),batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype('float32')
    np.savez_compressed(cache,embeddings=vec)
    print('Cached embeddings',cache)
vec=np.asarray(vec,dtype='float32')
assert vec.shape==(len(all_text),384)
id_to_ix=pd.Series(np.arange(len(all_text)),index=all_text.cve_id)
assert cohort.cve_id.isin(id_to_ix.index).all()
assert scoreboard.cve_id.isin(id_to_ix.index).all()


   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   --------------------------- ------------ 524.3/760.7 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 760.7/760.7 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/857.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/857.0 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.9 MB 1.5 MB/s eta 0:00:03
   -------- ------------------------------- 0.8/3.9 MB 1.5 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/3.9 MB 1.4 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/3.9 MB 1.4 MB/s eta 0:00:02
   ---------------- ---------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires pandas<3,>=1.4.0, but you have pandas 3.1.1 which is incompatible.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.


Batches:   0%|          | 0/2274 [00:00<?, ?it/s]

## 3. Time-aware nearest **already-known KEV** similarity

For each candidate snapshot, restrict neighbors to **KEV additions no later than its snapshot** (and published by the snapshot). This explicitly prevents a later KEV addition from serving as historical evidence. We retrieve the highest cosine similarity and mean of the three highest. Unlike the old neural head, we are not fitting 384-dimensional weights on a few dozen positives.


In [ ]:
kev_pool=kev.merge(vuln[['cve_id','published']],on='cve_id',how='left')
kev_pool=kev_pool[kev_pool.cve_id.isin(id_to_ix.index)].copy()

def neighbor_features(frame):
    # Calculate separately for each snapshot month to enforce the temporal KEV cutoff.
    result=np.zeros((len(frame),3),dtype='float32')
    for month,part in frame.groupby('snapshot_month',sort=True):
        snap=pd.Timestamp(month+'-01')+pd.offsets.MonthEnd()
        eligible=kev_pool[(kev_pool.date_added<=snap)&(kev_pool.published<=snap)]
        pos=part.index.to_numpy()
        if eligible.empty:
            continue
        q=vec[id_to_ix.loc[part.cve_id].to_numpy()]
        bank_ids=eligible.cve_id.tolist()
        b=vec[id_to_ix.loc[bank_ids].to_numpy()]
        # Chunk queries to avoid a huge matrix.
        for start in range(0,len(q),1024):
            stop=min(start+1024,len(q))
            sims=q[start:stop] @ b.T
            # Exclude same CVE if ever present due to unusual dates.
            for row_local,cve_id in enumerate(part.cve_id.iloc[start:stop]):
                matches=np.flatnonzero(eligible.cve_id.to_numpy()==cve_id)
                if len(matches): sims[row_local,matches]=-1
            best=np.sort(sims,axis=1)[:,-min(3,len(bank_ids)):]
            result[pos[start:stop],0]=best[:,-1]
            result[pos[start:stop],1]=best.mean(axis=1)
            result[pos[start:stop],2]=(sims>=0.80).sum(axis=1)
    return result

cohort=cohort.reset_index(drop=True)
scoreboard=scoreboard.reset_index(drop=True)
for frame in [cohort,scoreboard]:
    feats=neighbor_features(frame)
    frame[['kev_sim_max','kev_sim_top3','kev_sim_count_080']]=feats
print(cohort[['kev_sim_max','kev_sim_top3','kev_sim_count_080']].describe())


## 4. Frozen challenger rule + rolling quarters

We test *fixed* Granite-neighbor weights of 5%, 10%, 20%, instead of training new weights on the repeatedly inspected small validation cohort. The EPSS and nearest-KEV similarity are percentile ranked **within each exploit group**; all exploit-indicated CVEs remain above no-exploit CVEs. The nearest-KEV signal is new information and might outperform the baseline, but only validation can establish that.

This test is **development evidence**, not untouched confirmation: 2025 Q1–Q3 have previously influenced experimentation.


In [ ]:
def sorted_scores(frame,granite_weight=0):
    x=frame.reset_index(drop=True)
    result=np.empty(len(x),dtype=float)
    # Per-group percentile normalization.
    ep=np.zeros(len(x)); sim=np.zeros(len(x))
    for flag in [0,1]:
        indices=np.flatnonzero(x.any_exploit.to_numpy()==flag)
        if len(indices)==0:continue
        ep[indices]=rankdata(x.epss.to_numpy()[indices],method='average')/len(indices)
        sim[indices]=rankdata(x.kev_sim_top3.to_numpy()[indices],method='average')/len(indices)
    combo=(1-granite_weight)*ep+granite_weight*sim
    # Secondary tie breaks guarantee determinism.
    order=np.lexsort((x.cve_id.to_numpy(),-x.epss.to_numpy(),-combo,-x.any_exploit.to_numpy()))
    result[order]=np.arange(len(x),0,-1,dtype=float)
    return result

def report(frame,scores,model,fold):
    y=frame.label.to_numpy().astype(int)
    k=max(1,round(len(frame)*K_FRAC))
    hits=int(y[np.argsort(-scores,kind='stable')[:k]].sum())
    return {'fold':fold,'model':model,'n':len(frame),'positives':int(y.sum()),'k':k,'hits':hits,'AP':average_precision_score(y,scores),'ROC_AUC':roc_auc_score(y,scores) if y.min()!=y.max() else np.nan}

folds=[('2025-Q1','2025-01','2025-03'),('2025-Q2','2025-04','2025-06'),('2025-Q3','2025-07','2025-09')]
weights=[0,0.05,0.10,0.20]
records=[]
for label,start,end in folds:
    f=cohort[cohort.snapshot_month.between(start,end)].reset_index(drop=True)
    for w in weights:
        records.append(report(f,sorted_scores(f,w),f'neighbor {w:.0%}',label))
results=pd.DataFrame(records)
display(results)
summary=results.groupby('model').agg(total_hits=('hits','sum'),selected=('k','sum'),mean_AP=('AP','mean'),mean_ROC_AUC=('ROC_AUC','mean'))
summary['precision_at_k']=summary.total_hits/summary.selected
print('Pooled summary')
display(summary.sort_values(['total_hits','mean_AP'],ascending=False))
results.to_csv(OUT/'comparison.csv',index=False)


## 5. Baseline + challenger submissions (not claims of superiority)

The baseline CSV remains the default. A challenger file is generated for inspection, but **do not submit its ranking on the strength of a retrospective-description backtest**. Before deploying it, switch to original historical descriptions and retest.


In [ ]:
def export_top(frame,w,outname):
    scores=sorted_scores(frame,w)
    ranking=frame[['cve_id','snapshot_month','epss','any_exploit','kev_sim_max','kev_sim_top3','kev_sim_count_080']].copy()
    ranking['rank_score']=scores
    ranking=ranking.sort_values('rank_score',ascending=False)
    path=OUT/outname
    ranking[['cve_id']].head(500).to_csv(path,index=False)
    assert ranking[['cve_id']].head(500).cve_id.nunique()==500
    return ranking
baseline=export_top(scoreboard,0,'top500_baseline.csv')
challenger=export_top(scoreboard,0.10,'top500_granite_neighbors_EXPLORATORY.csv')
challenger.to_csv(OUT/'full_neighbor_ranking_EXPLORATORY.csv',index=False)
print('Wrote:',*[p.name for p in OUT.glob('top500*')])
print('Overlap in top 500:',len(set(baseline.cve_id.head(500))&set(challenger.cve_id.head(500))))
display(challenger.head(15))


## 6. What would make this a fair competition challenger?

1. **Historical text:** Obtain archived CVE JSON from official [CVEProject/cvelistV5 releases](https://github.com/CVEProject/cvelistV5) or commit history, and extract the description as of each CVE's `snapshot_month`. Its CNA description is preferable to retrospective CISA ADP additions. Also use contemporaneous text for historical KEV neighbor records.
2. **Time-restricted KEV pool:** Already enforced above for `date_added` and publication; keep it.
3. **Separate validation for model selection:** 2025 quarters were repeatedly inspected. After choosing a single fixed weight, run a new predeclared historical evaluation with untouched period or a genuinely new source.
4. **2026 feature parity:** The strict 2026 challenger must use archived text no later than each CVE's `snapshot_month`, and not later KEV additions. If unavailable, retain `top500_baseline.csv`.
5. **Compare the correct objective:** Hits@K at the competition selection ratio; AP secondary; no claim about beating the official **8/500** without official 2026 labels.

**Optional further novelty:** after temporal reconstruction, add publication-time vendor/product neighborhood features or contrastive self-supervised training (without future labels), but only if this basic neighbor signal shows lift.
